# Address Toolkit - Supporting AIMS/Package analysis

In [2]:
import pandas as pd
import os

In [3]:
base_location = r"Z:\AIMS\AIMS_Confidence_Score_Clerical\All_Source_Files\Address_Cleaning_Analysis"

# Input Data Quality

In [ ]:
address_cleaning_input_files = os.path.join(base_location, 'Input Data Processing', 'raw_input_files')
input_files = [f for f in os.listdir(address_cleaning_input_files) if f.endswith('.csv')]

In [ ]:
processing_stats = []
for file in input_files:
    file_path = os.path.join(address_cleaning_input_files, file)
    df = pd.read_csv(file_path)

    n_rows = len(df)
    n_cols = len(df.columns)
    cols = df.columns.tolist()
    nulls = df.isnull().sum()

    id_colname = cols[0]
    address_colname = cols[1]

    duplicates_id = df.duplicated(id_colname).sum()
    duplicates_address = df.duplicated(address_colname).sum()

    stats = {
        'file_name': file,
        'n_rows': n_rows,
        'n_cols': n_cols,
        'columns': cols,
        'duplicates_in_id_column': duplicates_id,
        'duplicates_in_address_column': duplicates_address
    }

    processing_stats.append(stats)

In [ ]:
series = [pd.DataFrame([stat]) for stat in processing_stats]
summary_df = pd.concat(series, ignore_index=True)
summary_df

In [ ]:
summary_df.to_csv(os.path.join(base_location, 'Input Data Processing','input_files_validation_summary.csv'), index=False)

# Input Data Standardisation

In [ ]:
standardisation_path = os.path.join(base_location, 'Input Data Processing', 'standardised_input_files')

In [ ]:
for file in input_files:
    file_path = os.path.join(address_cleaning_input_files, file)
    df = pd.read_csv(file_path)

    # Standardise Column Names
    df = df.rename(columns = {df.columns[0]: 'Query_ID', df.columns[1]: 'Supplied_Query_Address'})

    # Remove Duplicates based on Supplied_Query_Address
    df = df.drop_duplicates(subset=['Supplied_Query_Address'])

    # Save cleaned file
    cleaned_file_path = os.path.join(standardisation_path, file)
    df.to_csv(cleaned_file_path, index=False)

In [ ]:
# Validation of standardised files
processing_stats = []
for file in input_files:
    file_path = os.path.join(standardisation_path, file)
    df = pd.read_csv(file_path)

    n_rows = len(df)
    n_cols = len(df.columns)
    cols = df.columns.tolist()
    nulls = df.isnull().sum()

    id_colname = cols[0]
    address_colname = cols[1]

    duplicates_id = df.duplicated(id_colname).sum()
    duplicates_address = df.duplicated(address_colname).sum()

    stats = {
        'file_name': file,
        'n_rows': n_rows,
        'n_cols': n_cols,
        'columns': cols,
        'duplicates_in_id_column': duplicates_id,
        'duplicates_in_address_column': duplicates_address
    }

    processing_stats.append(stats)

In [ ]:
series = [pd.DataFrame([stat]) for stat in processing_stats]
summary_df = pd.concat(series, ignore_index=True)
summary_df

In [ ]:
summary_df.to_csv(os.path.join(base_location, 'Input Data Processing', 'standardised_input_files_validation_summary.csv'), index=False)

# Input Data Enrichment

In [ ]:
from pyspark.sql import SparkSession
from address_toolkit.validating import validate_from_list, validate_postcodes
from address_toolkit.resources import place_list, city_list, district_list, county_list, town_list, village_list, hamlet_list, suburb_list, bay_list

spark = SparkSession.builder.appName("AddressCleaningValidation").config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.cores", "4") \
    .getOrCreate()

In [ ]:
standardisation_path = os.path.join(base_location, 'Input Data Processing','standardised_input_files')
standardisation_files = os.listdir(standardisation_path)
standardisation_files = [f for f in standardisation_files if f.endswith('.csv')]

In [ ]:
for file in standardisation_files:

    df = spark.read.csv(os.path.join(standardisation_path, file), header=True, inferSchema=True)

    # Create Validation Flags
    df = validate_from_list(df, 'Supplied_Query_Address', 'place', place_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'town', town_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'village', village_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'hamlet', hamlet_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'bay', bay_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'suburb', suburb_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'city', city_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'district', district_list, 95, False)
    df = validate_from_list(df, 'Supplied_Query_Address', 'county', county_list, 95, False)
    df = validate_postcodes(df, 'Supplied_Query_Address')

    # Create Sum of Validation Flags
    df = df.withColumn('total_validated_flags',
                       df.validated_place_flag +
                       df.validated_town_flag +
                       df.validated_village_flag +
                       df.validated_hamlet_flag +
                       df.validated_bay_flag +
                       df.validated_suburb_flag +
                       df.validated_city_flag +
                       df.validated_district_flag +
                       df.validated_county_flag +
                       df.validated_postcode_flag)

    pandas_df = df.toPandas()
    pandas_df.to_csv(os.path.join(base_location, 'Input Data Processing', 'enriched_input_files', file), index=False)

# Cleaned Addresses

In [ ]:
enriched_files_path = os.path.join(base_location, 'Input Data Processing', 'enriched_input_files')
enriched_files = os.listdir(enriched_files_path)
enriched_files = [f for f in enriched_files if f.endswith('.csv')]

In [ ]:
from address_toolkit.cleaning import clean_punctuation, deduplicate_addresses, deduplicate_postcodes, standardise_street_types, rectify_postcodes, denoise_addresses
from address_toolkit.resources import consecutive_letters_regex

for file in enriched_files:

    df = spark.read.csv(os.path.join(enriched_files_path, file), header=True, inferSchema=True)

    # Create Sum of Validation Flags
    df = df.drop('total_validated_flags', 'validated_town_flag',
                        'validated_village_flag', 'validated_suburb_flag',
                        'validated_city_flag', 'validated_hamlet_flag', 'validated_county_flag', 'validated_postcode_flag')

    df = clean_punctuation(df, 'Supplied_Query_Address', create_flag = True, overwrite = True)
    df = deduplicate_addresses(df, 'Supplied_Query_Address', intracomponents = False, intercomponents = True, tolerance = 0, create_flag = True, overwrite = True)
    df = deduplicate_postcodes(df, 'Supplied_Query_Address', create_flag = True, overwrite = True)
    df = standardise_street_types(df, 'Supplied_Query_Address', create_flag = True, overwrite = True)
    df = rectify_postcodes(df, 'Supplied_Query_Address', create_flag = True, overwrite = True)
    df = denoise_addresses(df, 'Supplied_Query_Address', noise_pattern = consecutive_letters_regex, create_flag = True, overwrite = True)

    # Order columns
    df = df.select('Query_ID', 'Supplied_Query_Address',
                   'punctuation_cleaned_flag',
                   'deduplicated_address_flag',
                   'postcode_deduplicated_flag',
                   'street_type_standardised_flag',
                   'rectified_postcode_flag',
                   'noise_removed_flag')

    df_base = df.select('Query_ID', 'Supplied_Query_Address')

    pandas_df = df.toPandas()
    pandas_df_base = df_base.toPandas()

    pandas_df.to_csv(os.path.join(base_location, 'Functions Applied Address Data', 'functions_cleaned_files_enriched', file), index=False)
    pandas_df_base.to_csv(os.path.join(base_location, 'Functions Applied Address Data', 'functions_cleaned_files_base', file), index=False)

# Cleaned Addresses Processing Statistics

In [ ]:
cleaned_files_path = os.path.join(base_location, 'Functions Applied Address Data', 'functions_cleaned_files_enriched')
cleaned_files = os.listdir(cleaned_files_path)
cleaned_files = [f for f in cleaned_files if f.endswith('.csv')]

In [ ]:
breakdown_dfs = []
for file in cleaned_files:

    df = pd.read_csv(os.path.join(cleaned_files_path, file))
    df = df.drop(columns = ['Query_ID','Supplied_Query_Address'])

    stats = {}
    stats['file_name'] = file

    for col in df.columns:
        stats[col] = round(100 * df[col].sum() / len(df), 2)

    stats['total_rows'] = len(df)

    breakdown_dfs.append(pd.DataFrame([stats]))

breakdown_summary_df = pd.concat(breakdown_dfs, ignore_index=True)
breakdown_summary_df.to_csv(os.path.join(base_location, 'functions_cleaned_files_validation_summary.csv'), index=False)

# Batching Up Files (Functions Cleaned)

In [ ]:
import numpy as np

base_path = os.path.join(base_location, 'Functions Applied Address Data', 'functions_cleaned_files_base')
out_path = os.path.join(base_location, 'Functions Applied Address Data', 'functions_cleaned_files_to_aims')
for file in os.listdir(base_path):
    df = pd.read_csv(os.path.join(base_path, file))
    segments = [s for s in range(0, int(np.ceil(len(df) / 4000)))]

    segment = 0
    while segment < max(segments):

        start_segment = segment * 4000
        end_segment = (segment + 1) * 4000

        out_filename = file.replace('.csv', f"_BATCH{segment}.csv")
        batch_df = df[start_segment:end_segment]
        batch_df.to_csv(os.path.join(out_path, out_filename), index = False)

        segment = segment + 1

    # Final Batch
    out_filename = file.replace('.csv', f"_BATCH{segment}.csv")
    batch_df = df[(segment * 4000):]
    batch_df.to_csv(os.path.join(out_path, out_filename), index = False)


# Batching Up Files (Uncleaned)

In [ ]:
base_path = os.path.join(base_location, 'Input Data Processing', 'standardised_input_files')
out_path = os.path.join(base_location, 'Input Data Processing', 'input_files_to_aims')

for file in os.listdir(base_path):
    df = pd.read_csv(os.path.join(base_path, file))
    segments = [s for s in range(0, int(np.ceil(len(df) / 4000)))]

    segment = 0
    while segment < max(segments):

        start_segment = segment * 4000
        end_segment = (segment + 1) * 4000

        out_filename = file.replace('.csv', f"_BATCH{segment}.csv")
        batch_df = df[start_segment:end_segment]
        batch_df.to_csv(os.path.join(out_path, out_filename), index = False)

        segment = segment + 1

    # Final Batch
    out_filename = file.replace('.csv', f"_BATCH{segment}.csv")
    batch_df = df[(segment * 4000):]
    batch_df.to_csv(os.path.join(out_path, out_filename), index = False)

# Concatenating Batched Files (Post-AIMS)

In [4]:
cleaned_files_post_aims_path = os.path.join(base_location, 'AIMS Processing', 'aims_uprn_matched_historical_functions')
cleaned_files_post_aims_outpath = os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_aims_concatenated')
cleaned_files = os.listdir(cleaned_files_post_aims_path)

base_filenames = set([f.split('_BATCH')[0] for f in cleaned_files])
base_dictionary = {base_filename: [] for base_filename in base_filenames}

for file in cleaned_files:
    base_filename = file.split('_BATCH')[0]
    base_dictionary[base_filename].append(file)

for base_file in base_dictionary:
    batch_files = base_dictionary[base_file]
    batch_dfs = []
    for batch_file in batch_files:
        df = pd.read_csv(os.path.join(cleaned_files_post_aims_path, batch_file))
        batch_dfs.append(df)

    concatenated_df = pd.concat(batch_dfs, ignore_index=True)
    outname = base_file + '.csv'
    concatenated_df.to_csv(os.path.join(cleaned_files_post_aims_outpath, outname), index=False)

In [5]:
baseline_files_post_aims_path = os.path.join(base_location, 'AIMS Processing', 'aims_uprn_matched_historical_baseline')
baseline_files_post_aims_outpath = os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'baseline_aims_concatenated')
baseline_files = os.listdir(baseline_files_post_aims_path)

base_filenames = set([f.split('_BATCH')[0] for f in baseline_files])
base_dictionary = {base_filename: [] for base_filename in base_filenames}

for file in baseline_files:
    base_filename = file.split('_BATCH')[0]
    base_dictionary[base_filename].append(file)

for base_file in base_dictionary:
    batch_files = base_dictionary[base_file]
    batch_dfs = []
    for batch_file in batch_files:
        df = pd.read_csv(os.path.join(baseline_files_post_aims_path, batch_file))
        batch_dfs.append(df)

    concatenated_df = pd.concat(batch_dfs, ignore_index=True)
    outname = base_file + '.csv'
    concatenated_df.to_csv(os.path.join(baseline_files_post_aims_outpath, outname), index=False)

# Enriching Post-AIMS processed files

In [6]:
baseline_files_concatenated_path = os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'baseline_aims_concatenated')
baseline_files_enriched_pre_aims = os.path.join(base_location, 'Input Data Processing', 'enriched_input_files')
baseline_files_enriched_path = os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'baseline_aims_enriched')

processing_stats = []
for file in os.listdir(baseline_files_concatenated_path):

    post_aims = pd.read_csv(os.path.join(baseline_files_concatenated_path, file))
    post_aims = post_aims.rename(columns = {'id':'Query_ID',
                                            'uprn':'baseline_uprn',
                                            'inputAddress':'Supplied_Query_Address',
                                            'confidenceScore':'confidenceScoreBaseline'})
    post_aims = post_aims[['Query_ID', 'Supplied_Query_Address', 'baseline_uprn', 'confidenceScoreBaseline']]

    pre_aims = pd.read_csv(os.path.join(baseline_files_enriched_pre_aims, file))
    pre_aims = pre_aims.drop(columns = ['Supplied_Query_Address'])

    merged = pre_aims.merge(post_aims, on = 'Query_ID', how = 'left')

    # Save enriched baseline file
    merged.to_csv(os.path.join(baseline_files_enriched_path, file), index = False)

    # Log processing statistics
    stats = {
        'file_name': file,
        'input_data_records': len(pre_aims),
        'data_loss': len(pre_aims) - len(post_aims)
    }

    processing_stats.append(stats)

In [7]:
processing_stats = pd.DataFrame(processing_stats)
processing_stats.to_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'baseline_aims_enrichment_processing_stats.csv'), index=False)

In [8]:
cleaned_files_concatenated_path = os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_aims_concatenated')
cleaned_files_enriched_pre_aims = os.path.join(base_location, 'Functions Applied Address Data' ,'functions_cleaned_files_enriched')
cleaned_files_enriched_path = os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_aims_enriched')

processing_stats = []
for file in os.listdir(cleaned_files_concatenated_path):

    post_aims = pd.read_csv(os.path.join(cleaned_files_concatenated_path, file))
    post_aims = post_aims.rename(columns = {'id':'Query_ID',
                                            'uprn':'functions_uprn',
                                            'inputAddress':'Cleaned_Query_Address',
                                            'confidenceScore':'confidenceScoreFunctions'})
    post_aims = post_aims[['Query_ID', 'functions_uprn', 'Cleaned_Query_Address', 'confidenceScoreFunctions']]

    pre_aims = pd.read_csv(os.path.join(cleaned_files_enriched_pre_aims, file))
    pre_aims = pre_aims.drop(columns = ['Supplied_Query_Address'])

    merged = pre_aims.merge(post_aims, on = 'Query_ID', how = 'left')

    # Save enriched functions file
    merged.to_csv(os.path.join(cleaned_files_enriched_path, file), index = False)

    # Log processing statistics
    stats = {
        'file_name': file,
        'input_data_records': len(pre_aims),
        'data_loss': len(pre_aims) - len(post_aims)
    }

    processing_stats.append(stats)

In [9]:
processing_stats = pd.DataFrame(processing_stats)
processing_stats.to_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_aims_enrichment_processing_stats.csv'), index=False)

# Analysis Data Preparation

## Joining Baseline Results to Functions Cleaned Results

In [12]:
for file in os.listdir(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_aims_enriched')):
    if file.endswith('.csv'):
        aims_functions_enriched = pd.read_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_aims_enriched', file))
        baseline_aims_data = pd.read_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'baseline_aims_enriched', file))

        # Merge AIMS functions enriched data with baseline AIMS data
        merged_aims_data = pd.merge(aims_functions_enriched, baseline_aims_data, on=['Query_ID'], how='left')

        # Sort out column ordering
        primary_columns = ['Query_ID', 'Supplied_Query_Address', 'Cleaned_Query_Address', 'baseline_uprn', 'functions_uprn', 'confidenceScoreBaseline', 'confidenceScoreFunctions']
        uneeded_columns = ['matchedAddress', 'matchType', 'documentScore', 'rank', 'addressType(Paf/Nag)', 'airRating']
        secondary_columns = [col for col in merged_aims_data.columns if col not in primary_columns + uneeded_columns]
        merged_aims_data = merged_aims_data[primary_columns + secondary_columns]

        # Write merged data to file
        merged_aims_data.to_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_aims_enriched', file), index=False)


## Joining Clerical Results

In [13]:
processing_stats = []
for file in os.listdir(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_aims_enriched')):
    if file.endswith('.csv'):

        merged_aims_data = pd.read_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_aims_enriched', file))
        clerical_review_data = pd.read_csv(os.path.join(base_location, 'AIMS Processing', 'aims_uprn_matched_historical_clerical', file))

        # Renaming Clerical Data Columns to avoid confusion when merging
        clerical_review_data = clerical_review_data.rename(columns = {clerical_review_data.columns[0]: 'Query_ID',
                                                                    clerical_review_data.columns[1]: 'Standard_Query_Address',
                                                                    'CLERICAL_UPRN': 'clerical_uprn'})

        # From clerical data, we only need the Query_ID and the clerical_uprn. The confidence score would be determined by aims.
        clerical_review_data = clerical_review_data[['Query_ID', 'clerical_uprn']]

        # Merge together
        final_merged_data = pd.merge(merged_aims_data, clerical_review_data, on=['Query_ID'], how='left')

        # Note data loss
        data_loss_functions = final_merged_data['Cleaned_Query_Address'].isnull().sum()
        data_loss_baseline = final_merged_data['Supplied_Query_Address'].isnull().sum()
        data_loss_clerical = final_merged_data['clerical_uprn'].isnull().sum()
        file_stats = {
            'file_name': file,
            'expected_records': len(final_merged_data),
            'data_loss_functions': data_loss_functions,
            'data_loss_baseline': data_loss_baseline,
            'data_loss_clerical': data_loss_clerical}
        processing_stats.append(file_stats)

        # Sort Out Column Ordering
        primary_columns = ['Query_ID', 'Supplied_Query_Address', 'Cleaned_Query_Address', 'baseline_uprn', 'functions_uprn', 'clerical_uprn', 'confidenceScoreBaseline', 'confidenceScoreFunctions']
        secondary_columns = [col for col in final_merged_data.columns if col not in primary_columns]
        final_merged_data = final_merged_data[primary_columns + secondary_columns]

        # Write merged data to file
        final_merged_data.to_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_clerical_aims_enriched', file), index=False)


In [14]:
processing_df = pd.DataFrame(processing_stats)
processing_df.to_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_clerical_aims_enriched_validation_summary.csv'), index=False)

# Analysis Data (UPRN Change Type, Confidence Score Change)

In [15]:
def uprn_category(baseline_uprn, functions_uprn, actual_uprn):
    if [baseline_uprn, functions_uprn, actual_uprn].count(0) > 0:
        return 'Unknown'
    if baseline_uprn == actual_uprn == functions_uprn:
        return 'Aligned'
    if baseline_uprn != actual_uprn and functions_uprn == actual_uprn:
        return 'Corrected'
    if baseline_uprn == actual_uprn and functions_uprn != actual_uprn:
        return 'Malformed'
    if (baseline_uprn == functions_uprn and baseline_uprn != actual_uprn) | (baseline_uprn != functions_uprn and functions_uprn != actual_uprn and baseline_uprn != actual_uprn):
        return 'Misaligned'

In [17]:
for file in os.listdir(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_clerical_aims_enriched')):
    if file.endswith('.csv'):

        final_merged_data = pd.read_csv(os.path.join(base_location, 'Analysis Data Preparation (Historical)', 'functions_baseline_clerical_aims_enriched', file))
        final_merged_data['confidenceScoreDifference'] = final_merged_data['confidenceScoreFunctions'] - final_merged_data['confidenceScoreBaseline']

        # Dropping all NULLS where data loss has occured after joining with AIMS and Clerical data as we cannot determine confidence score differences or UPRN alignment for these records
        final_merged_data = final_merged_data.dropna(subset = ['Cleaned_Query_Address', 'Supplied_Query_Address', 'clerical_uprn'])

        # Applying integer type to UPRN columns for easier comparison
        final_merged_data['baseline_uprn'] = final_merged_data['baseline_uprn'].astype(int)
        final_merged_data['functions_uprn'] = final_merged_data['functions_uprn'].astype(int)
        final_merged_data['clerical_uprn'] = final_merged_data['clerical_uprn'].astype(int)

        final_merged_data['uprn_category'] = final_merged_data.apply(lambda row: uprn_category(row['baseline_uprn'], row['functions_uprn'], row['clerical_uprn']), axis=1)

        # Save output for analysis
        final_merged_data.to_csv(os.path.join(base_location, 'Analysis Data', 'All Files', file), index=False)

# Concatenate by Source Type

In [18]:
data_source_dictionary = {'Electoral Register': ['RETURN_FILES_Electoral_Register_55to64_6K_ADDRESS_CLEANING_INPUT.csv', 'RETURN_FILES_Electoral_Register_2020_65to100_10K_ADDRESS_CLEANING_INPUT.csv'],
                        'Land Registry': ['RETURN_FILES_Land_Registry_Prices_Paid_55to64_100K_ADDRESS_CLEANING_INPUT.csv', 'RETURN_FILES_Land_Registry_Prices_Paid_65to100_10K_ADDRESS_CLEANING_INPUT.csv'],
                        'PDS Stocks': ['RETURN_FILES_PDS_Stocks_55to64_8K_ADDRESS_CLEANING_INPUT.csv', 'RETURN_FILES_PDS_Stocks_2022_65to89_20K_ADDRESS_CLEANING_INPUT.csv', 'RETURN_FILES_PDS_Stocks_2022_90to100_11K_ADDRESS_CLEANING_INPUT.csv'],
                        'TDP': ['RETURN_FILES_TDP_55to64_20K_ADDRESS_CLEANING_INPUT.csv', 'RETURN_FILES_TDP_65to100_10K_ADDRESS_CLEANING_INPUT.csv']}

In [19]:
for source_type in data_source_dictionary:
    sources = data_source_dictionary[source_type]
    all_data = []
    for source in sources:
        data = pd.read_csv(os.path.join(base_location, 'Analysis Data', 'All Files', source))
        all_data.append(data)

    all_data = pd.concat(all_data, ignore_index=True)
    all_data.to_csv(os.path.join(base_location, 'Analysis Data', 'Categorised', source_type + '.csv'), index=False)

# Analysis

## UPRN Category Breakdown Against Address Source

In [20]:
results = []
for file in os.listdir(os.path.join(base_location, 'Analysis Data', 'Categorised')):
    df = pd.read_csv(os.path.join(base_location, 'Analysis Data', 'Categorised', file))

    # Calculate distribution of UPRN categories
    aligned = len(df[df['uprn_category'] == 'Aligned'])
    misaligned = len(df[df['uprn_category'] == 'Misaligned'])
    corrected = len(df[df['uprn_category'] == 'Corrected'])
    malformed = len(df[df['uprn_category'] == 'Malformed'])
    unknown = len(df[df['uprn_category'] == 'Unknown'])

    # Store Statistics
    stats = {
        'Address Source': file.replace('.csv', ''),
        'Aligned': aligned,
        'Misaligned': misaligned,
        'Corrected': corrected,
        'Malformed': malformed,
        'Unknown': unknown}

    results.append(stats)

# Create Statistics Dataframe
results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'address_source_uprn_breakdown.csv'), index=False)

## UPRN Category Breakdown Against Flag Impact

In [21]:
flag_columns = ['punctuation_cleaned_flag', 'deduplicated_address_flag', 'postcode_deduplicated_flag', 'street_type_standardised_flag', 'rectified_postcode_flag', 'noise_removed_flag']
uprn_outcomes = ['Aligned', 'Misaligned', 'Corrected', 'Malformed']

stats = []
for flag in flag_columns:
    flag_stats = {}
    flag_stats['flag'] = flag

    # Initialise flag_stats for each UPRN Outcome
    for outcome in uprn_outcomes:
        flag_stats[outcome] = 0

    # Loop through files and sum up flag_stats for each UPRN outcome where flag is 1
    for file in os.listdir(os.path.join(base_location, 'Analysis Data', 'Categorised')):
        df = pd.read_csv(os.path.join(base_location, 'Analysis Data', 'Categorised', file))
        df = df[df[flag] == 1]

        # Update the running record of statistics for each UPRN Category
        for outcome in uprn_outcomes:
            sliced = df[df['uprn_category'] == outcome]
            flag_stats[outcome] = flag_stats[outcome] + len(sliced)

    # Store statistics
    stats.append(flag_stats)

# Create Dataframe of Statistics
statistics_df = pd.DataFrame(stats)
statistics_df.to_csv(os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'flag_uprn_breakdown.csv'), index=False)

## UPRN Category Breakdown Against Confidence Scores

In [22]:
statistics = []
for outcome in uprn_outcomes:
    data = []
    for file in os.listdir(os.path.join(base_location, 'Analysis Data', 'Categorised')):
        # Read Dataframe, filter to the UPRN Category
        df = pd.read_csv(os.path.join(base_location, 'Analysis Data', 'Categorised', file))
        sliced = df[df['uprn_category'] == outcome]
        # Store the sliced dataframe
        data.append(sliced)

    # Calculate Statistics
    all_outcome_data = pd.concat(data, ignore_index=True)
    mean = all_outcome_data['confidenceScoreDifference'].mean()
    median = all_outcome_data['confidenceScoreDifference'].median()
    std = all_outcome_data['confidenceScoreDifference'].std()
    minimum = all_outcome_data['confidenceScoreDifference'].min()
    maximum = all_outcome_data['confidenceScoreDifference'].max()

    # Store Statistics
    stats = {'UPRN Category': outcome,
             '# Addresses': len(all_outcome_data),
            'mean_confidence_score_difference': round(mean, 2),
            'median_confidence_score_difference': round(median, 2),
            'std_confidence_score_difference': round(std, 2),
            'min_confidence_score_difference': round(minimum, 2),
            'max_confidence_score_difference': round(maximum, 2)}

    statistics.append(stats)

# Create Dataframe of Statistics
statistics_df = pd.DataFrame(statistics)
statistics_df.to_csv(os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'confidence_score_uprn_breakdown.csv'), index=False)

## UPRN Category Breakdown Against Flag Impact per Address Source

In [23]:
flag_columns = ['punctuation_cleaned_flag', 'deduplicated_address_flag', 'postcode_deduplicated_flag', 'street_type_standardised_flag', 'rectified_postcode_flag', 'noise_removed_flag']
uprn_outcomes = ['Aligned', 'Misaligned', 'Corrected', 'Malformed']

for file in os.listdir(os.path.join(base_location, 'Analysis Data', 'Categorised')):
    df = pd.read_csv(os.path.join(base_location, 'Analysis Data', 'Categorised', file))

    flag_stats = []
    # Loop through each flag
    for flag in flag_columns:
        flag_data = {}
        flag_data['flag'] = flag

        # Update the running total of statistics for each outcome
        for outcome in uprn_outcomes:
            df_slice = df[(df['uprn_category'] == outcome) & (df[flag] == 1)]
            flag_data[outcome] = len(df_slice)

        flag_stats.append(flag_data)

    # Create dataframe of statistics for each Address Source
    flag_stats_df = pd.DataFrame(flag_stats)
    flag_stats_df.to_csv(os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'Flag Impact and UPRN Category Breakdown per Address Source', file), index=False)


## UPRN Category Breakdown Against Confidence Scores per Address Source

In [24]:
for file in os.listdir(os.path.join(base_location, 'Analysis Data', 'Categorised')):
    df = pd.read_csv(os.path.join(base_location, 'Analysis Data', 'Categorised', file))

    statistics = []
    for outcome in uprn_outcomes:

        # Slice dataframe and calculate statistics
        df_slice = df[(df['uprn_category'] == outcome)]
        mean = df_slice['confidenceScoreDifference'].mean()
        median = df_slice['confidenceScoreDifference'].median()
        std = df_slice['confidenceScoreDifference'].std()
        minimum = df_slice['confidenceScoreDifference'].min()
        maximum = df_slice['confidenceScoreDifference'].max()

        # Store statistics
        stats = {
            'UPRN Category': outcome,
            '# Addresses': len(df_slice),
            'mean_confidence_score_difference': round(mean, 2),
            'median_confidence_score_difference': round(median, 2),
            'std_confidence_score_difference': round(std, 2),
            'min_confidence_score_difference': round(minimum, 2),
            'max_confidence_score_difference': round(maximum, 2)}

        statistics.append(stats)

    # Create dataframe of statistics for each Address Source
    statistics_df = pd.DataFrame(statistics)
    statistics_df.to_csv(os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'Confidence Score and UPRN Category Breakdown per Address Source', file), index=False)

## Input Data Address Condition

In [31]:
enriched_analysis_path = os.path.join(base_location, 'Analysis Data', 'Categorised')
validation_flags = ['validated_place_flag', 'validated_town_flag', 'validated_village_flag', 'validated_hamlet_flag', 'validated_bay_flag', 'validated_suburb_flag', 'validated_city_flag', 'validated_district_flag', 'validated_county_flag', 'validated_postcode_flag']
out_location_validated = os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'Address Locality Impact and UPRN Category Breakdown per Address Source', 'All Flags', 'Validated')
out_location_invalidated = os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'Address Locality Impact and UPRN Category Breakdown per Address Source', 'All Flags', 'Invalidated')


# Loop through files
for file in os.listdir(enriched_analysis_path):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(enriched_analysis_path, file))
        results = []

        # Check distributions across those flags which are validated
        for validated_flag in validation_flags:
            df_slice = df[df[validated_flag] == 1]
            storage = {}
            storage['flag'] = validated_flag
            storage['# Addresses'] = len(df_slice)
            # Loop through UPRN outcomes
            for outcome in ['Aligned', 'Misaligned', 'Corrected', 'Malformed']:
                outcome_slice = df_slice[df_slice['uprn_category'] == outcome]
                storage[outcome] = len(outcome_slice)

            results.append(storage)
        results_df = pd.DataFrame(results)
        results_df.to_csv(os.path.join(out_location_validated, file), index=False)

# Loop through files
for file in os.listdir(enriched_analysis_path):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(enriched_analysis_path, file))
        results = []

        # Check distributions across those flags which are invalidated
        for validated_flag in validation_flags:
            df_slice = df[df[validated_flag] == 0]
            storage = {}
            storage['flag'] = validated_flag
            storage['# Addresses'] = len(df_slice)
            # Loop through UPRN outcomes
            for outcome in ['Aligned', 'Misaligned', 'Corrected', 'Malformed']:
                outcome_slice = df_slice[df_slice['uprn_category'] == outcome]
                storage[outcome] = len(outcome_slice)

            results.append(storage)
        results_df = pd.DataFrame(results)
        results_df.to_csv(os.path.join(out_location_invalidated, file), index=False)


## UPRN Breakdown against Address Locality per Address Source

In [33]:
for file in os.listdir(out_location_validated):
    if file.endswith('.csv'):
        validated_df = pd.read_csv(os.path.join(out_location_validated, file))
        invalidated_df = pd.read_csv(os.path.join(out_location_invalidated, file))

        merged_df = pd.merge(validated_df, invalidated_df, on='flag', suffixes=('_validated', '_invalidated'))

        for outcome in ['Aligned', 'Misaligned', 'Corrected', 'Malformed']:
            merged_df[outcome + '_validated_pct'] = round(100 * merged_df[outcome + '_validated'] / merged_df['# Addresses_validated'], 2)
            merged_df[outcome + '_invalidated_pct'] = round(100 * merged_df[outcome + '_invalidated'] / merged_df['# Addresses_invalidated'], 2)

        # Filter Columns to Outcome Differences Only
        difference_columns = ['flag'] + [col for col in merged_df.columns if '_pct' in col] + ['# Addresses_validated', '# Addresses_invalidated']
        merged_df = merged_df[difference_columns]

        merged_df.to_csv(os.path.join(base_location, 'Analysis Data', 'UPRN Breakdown', 'Address Locality Impact and UPRN Category Breakdown per Address Source', 'All Flags', 'Validated vs Invalidated', file), index=False)


## Binned Confidence Score (Min, Median, Mean, Max, StDev) per Address Source

In [35]:
import numpy as np

data_path = os.path.join(base_location, 'Analysis Data', 'Categorised')
out_path = os.path.join(base_location, 'Analysis Data', 'Binned Confidence Scores', 'Overall Statistics per Address Source')

for file in os.listdir(data_path):
    df = pd.read_csv(os.path.join(data_path, file))
    df = df[df['uprn_category']=='Aligned']
    df['bin'] = df['confidenceScoreBaseline'].apply(lambda x: np.floor(x))

    minimum = df.groupby('bin')['confidenceScoreDifference'].min().reset_index()
    minimum = minimum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMinimum'})
    maximum = df.groupby('bin')['confidenceScoreDifference'].max().reset_index()
    maximum = maximum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMaximum'})
    median = df.groupby('bin')['confidenceScoreDifference'].median().reset_index()
    median = median.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMedian'})
    mean = df.groupby('bin')['confidenceScoreDifference'].mean().reset_index()
    mean = mean.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMean'})
    stdev = df.groupby('bin')['confidenceScoreDifference'].std().reset_index()
    stdev = stdev.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceStDev'})
    counts = df.groupby('bin')['confidenceScoreDifference'].count().reset_index()
    counts = counts.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceCounts'})

    merge_in_loop = [median, mean, stdev, counts]
    df = pd.merge(minimum, maximum, how = 'left', on = 'bin')
    for result in merge_in_loop:
        df = pd.merge(df, result, how = 'left', on = 'bin')

    df.to_csv(os.path.join(out_path, file), index = False)


## Binned Confidence Scores against Flag per Address Source

In [36]:
flag_columns = ['punctuation_cleaned_flag', 'deduplicated_address_flag', 'postcode_deduplicated_flag', 'street_type_standardised_flag', 'rectified_postcode_flag', 'noise_removed_flag']
out_path = os.path.join(base_location, 'Analysis Data', 'Binned Confidence Scores', 'Flag Breakdown per Address Source')

for file in os.listdir(data_path):

    base_df = pd.read_csv(os.path.join(data_path, file))
    base_df = base_df[base_df['uprn_category']=='Aligned']
    base_df['bin'] = base_df['confidenceScoreBaseline'].apply(lambda x: np.floor(x))

    for flag in flag_columns:
        df = base_df[base_df[flag]==1]
        minimum = df.groupby('bin')['confidenceScoreDifference'].min().reset_index()
        minimum = minimum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMinimum'})
        maximum = df.groupby('bin')['confidenceScoreDifference'].max().reset_index()
        maximum = maximum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMaximum'})
        median = df.groupby('bin')['confidenceScoreDifference'].median().reset_index()
        median = median.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMedian'})
        mean = df.groupby('bin')['confidenceScoreDifference'].mean().reset_index()
        mean = mean.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMean'})
        stdev = df.groupby('bin')['confidenceScoreDifference'].std().reset_index()
        stdev = stdev.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceStDev'})
        counts = df.groupby('bin')['confidenceScoreDifference'].count().reset_index()
        counts = counts.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceCounts'})

        merge_in_loop = [median, mean, stdev, counts]
        df = pd.merge(minimum, maximum, how = 'left', on = 'bin')
        for result in merge_in_loop:
            df = pd.merge(df, result, how = 'left', on = 'bin')

        folder_path = os.path.join(out_path, file.replace('.csv',''))
        if not os.path.exists(folder_path):
            os.makedirs(os.path.join(folder_path))

        filename = file.replace('.csv','') + '_' + flag + '.csv'
        df.to_csv(os.path.join(folder_path, filename), index = False)

## Binned Confidence Scores (All Sources)

In [37]:
out_path = os.path.join(base_location, 'Analysis Data', 'Binned Confidence Scores', 'overall.csv')
all_results = []
for file in os.listdir(data_path):

    base_df = pd.read_csv(os.path.join(data_path, file))
    base_df = base_df[base_df['uprn_category']=='Aligned']
    all_results.append(base_df)

df = pd.concat(all_results)
df['bin'] = df['confidenceScoreBaseline'].apply(lambda x: np.floor(x))

minimum = df.groupby('bin')['confidenceScoreDifference'].min().reset_index()
minimum = minimum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMinimum'})
maximum = df.groupby('bin')['confidenceScoreDifference'].max().reset_index()
maximum = maximum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMaximum'})
median = df.groupby('bin')['confidenceScoreDifference'].median().reset_index()
median = median.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMedian'})
mean = df.groupby('bin')['confidenceScoreDifference'].mean().reset_index()
mean = mean.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMean'})
stdev = df.groupby('bin')['confidenceScoreDifference'].std().reset_index()
stdev = stdev.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceStDev'})
counts = df.groupby('bin')['confidenceScoreDifference'].count().reset_index()
counts = counts.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceCounts'})

merge_in_loop = [median, mean, stdev, counts]
df = pd.merge(minimum, maximum, how = 'left', on = 'bin')
for result in merge_in_loop:
    df = pd.merge(df, result, how = 'left', on = 'bin')

df.to_csv(out_path, index = False)

## Binned Confidence Scores against Flag Breakdown (All Sources)

In [38]:
flag_columns = ['punctuation_cleaned_flag', 'deduplicated_address_flag', 'postcode_deduplicated_flag', 'street_type_standardised_flag', 'rectified_postcode_flag', 'noise_removed_flag']
out_path = os.path.join(base_location, 'Analysis Data', 'Binned Confidence Scores', 'Overall Statistics per Flag')

for flag in flag_columns:
    all_results = []

    for file in os.listdir(data_path):
        base_df = pd.read_csv(os.path.join(data_path, file))
        base_df = base_df[base_df['uprn_category']=='Aligned']
        base_df = base_df[base_df[flag]==1]
        all_results.append(base_df)

    df = pd.concat(all_results)
    df['bin'] = df['confidenceScoreBaseline'].apply(lambda x: np.floor(x))

    minimum = df.groupby('bin')['confidenceScoreDifference'].min().reset_index()
    minimum = minimum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMinimum'})
    maximum = df.groupby('bin')['confidenceScoreDifference'].max().reset_index()
    maximum = maximum.rename(columns = {'confidenceScoreDifference': 'ConfidenceScoreDifferenceMaximum'})
    median = df.groupby('bin')['confidenceScoreDifference'].median().reset_index()
    median = median.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMedian'})
    mean = df.groupby('bin')['confidenceScoreDifference'].mean().reset_index()
    mean = mean.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceMean'})
    stdev = df.groupby('bin')['confidenceScoreDifference'].std().reset_index()
    stdev = stdev.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceStDev'})
    counts = df.groupby('bin')['confidenceScoreDifference'].count().reset_index()
    counts = counts.rename(columns = {'confidenceScoreDifference':'ConfidenceScoreDifferenceCounts'})

    merge_in_loop = [median, mean, stdev, counts]
    df = pd.merge(minimum, maximum, how = 'left', on = 'bin')
    for result in merge_in_loop:
        df = pd.merge(df, result, how = 'left', on = 'bin')

    out_location = os.path.join(out_path, flag + '.csv')
    df.to_csv(out_location, index = False)